In [1]:
import torch
from torch import nn
import torch.nn.functional as F

import re
import unicodedata

In [12]:
class Language:
    def __init__(self,name):
        self.words2index = {"SOS":0,"EOS":1,"UNK":2}
        self.name = name
        self.index2words = {0:"SOS",1:"EOS",2:"UNK"}
        self.word_count = {}
        self.n_words = 3
    def add_word(self,word):
        if word not in self.word_count.keys():
            self.words2index[word] = self.n_words
            self.index2words[self.n_words] = word
            self.word_count[word] = 1
            self.n_words += 1
        else:
            self.word_count[word] = 1
    
    def add_sentence(self,sentence):
        for word in sentence.split(" "):
            self.add_word(word)
    
    def encode(self,sentence):
        encoded_rep = list()
        for word in sentence.split(" "):
            if word not in self.word_count.keys():
                encoded_rep.append(self.words2index["UNK"])
            else:
                encoded_rep.append(self.words2index[word])
        return encoded_rep
    
    def decode(self,sentence):
        return "".join([self.index2words[index] for index in sentence.split(" ")])

In [8]:
def unicodeToASCII(unicode):
    return ''.join(
        c for c in unicodedata.normalize('NFD',unicode)
        if unicodedata.category(c) != 'Mn'
    )
def normaliseText(text):
    # regex to remove punctuations
    text = re.sub(r"([.!?])",r" \1",text)
    text = unicodeToASCII(text.lower().strip())
    text = re.sub(r"[^a-zA-Z!?]+", r" ", text)
    return text.lower().strip()

def filterPair(p):
    return len(p[0].split(' ')) < 20 and \
        len(p[1].split(' ')) < 20 

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [13]:
def readData():
    data = open("./data/fra.txt","r").read().splitlines()
    pairs = [[normaliseText(s) for s in l.split('\t')] for l in data]
    pairs = filterPairs(pairs)
    
    french = Language("French")
    english = Language("English")

    return english,french,pairs

In [14]:
english,french,pairs = readData()

In [16]:
for english_translation,french_translation,_ in pairs:
    english.add_sentence(english_translation)
    french.add_sentence(french_translation)

In [21]:
print(f"Number of english words: {english.n_words}")
print(f"Number of french words: {french.n_words}")

Number of english words: 15877
Number of french words: 25466


[['go', 'va !', 'cc by france attribution tatoeba org cm wittydev'],
 ['go', 'marche', 'cc by france attribution tatoeba org cm micsmithel'],
 ['go', 'en route !', 'cc by france attribution tatoeba org cm felix'],
 ['go', 'bouge !', 'cc by france attribution tatoeba org cm micsmithel'],
 ['hi', 'salut !', 'cc by france attribution tatoeba org cm aiji'],
 ['hi', 'salut', 'cc by france attribution tatoeba org cm gillux'],
 ['run !',
  'cours !',
  'cc by france attribution tatoeba org papabear sacredceltic'],
 ['run !',
  'courez !',
  'cc by france attribution tatoeba org papabear sacredceltic'],
 ['run !',
  'prenez vos jambes a vos cous !',
  'cc by france attribution tatoeba org papabear sacredceltic'],
 ['run !',
  'file !',
  'cc by france attribution tatoeba org papabear sacredceltic'],
 ['run !',
  'filez !',
  'cc by france attribution tatoeba org papabear sacredceltic'],
 ['run !',
  'cours !',
  'cc by france attribution tatoeba org papabear franlexcois'],
 ['run !',
  'fuyez 